In [0]:
# Store the values from the text input widgets into variables
my_catalog = dbutils.widgets.get("catalog")
my_schema = dbutils.widgets.get("schema")

# Set path to your volume
my_volume_path = f"/Volumes/{my_catalog}/{my_schema}/retail_data"

# display the variables
print(f"my_catalog: {my_catalog}")
print(f"my_schema: {my_schema}")
print(f"my_volume_path: {my_volume_path}")

In [0]:
# -- 1. Drop the table if it exists
spark.sql(f"DROP TABLE IF EXISTS {my_catalog}.{my_schema}.tb_customer_sales_silver")

# -- 2. Create the table
query = f"""
CREATE TABLE {my_catalog}.{my_schema}.tb_customer_sales_silver
USING DELTA
AS
SELECT
  c.customer_id,
  concat(c.first_name,' ',c.last_name) as customer_name,
  c.email,
  c.country,
  c.segment,
  c.is_active,
  o.order_id,
  o.order_date,
  o.status as order_status,
  o.payment_method,
  o.shipping_fee,
  o.order_total,
  s.sale_id,
  s.product_id,
  s.product_name,
  s.category,
  s.quantity,
  s.unit_price,
  s.discount_pct,
  s.line_total,
  s.sale_date
FROM
  {my_catalog}.{my_schema}.tb_customers_bronze as c
  INNER JOIN {my_catalog}.{my_schema}.tb_orders_bronze as o
    ON c.customer_id = o.customer_id
  INNER JOIN {my_catalog}.{my_schema}.tb_sales_bronze as s
    ON o.order_id = s.order_id
    ;
"""

print(query)

# -- 3. Execute the query
spark.sql(query)

In [0]:
## Read the table into a dataframe
df = (
    spark
    .sql(f"""
            SELECT * FROM {my_catalog}.{my_schema}.tb_customer_sales_silver
        """)
    )

# check for duplicate records
duplicate_exists = df.count() > df.dropDuplicates().count()

# Set boolean flag in task values
dbutils.jobs.taskValues.set(key="has_duplicates", value=duplicate_exists)